# ReAct: Synergizing Reasoning and Acting in Language Models

## Learning Objectives
1. Understand the ReAct loop: thought-action-observation-repeat
2. Design tool-use agents with multiple actions
3. Implement error recovery and loop termination
4. Compare pure reasoning (CoT) vs reasoning+acting (ReAct)
5. Analyze agent trajectories to identify bottlenecks

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import re
from collections import defaultdict
from typing import Dict, List, Tuple

# Device and reproducibility
np.random.seed(42)

print("ReAct Agent Implementation Notebook")
print("=" * 50)

## Level 1: Basic ReAct Loop

Simplest agent: thought → action → observation → repeat

In [ ]:
class SimpleReActAgent:
    """Minimal ReAct agent for demonstration."""
    
    def __init__(self, tools: Dict, max_iterations: int = 5):
        self.tools = tools
        self.max_iterations = max_iterations
    
    def run(self, question: str, simulated_thoughts: List[str], 
            simulated_actions: List[str]) -> Dict:
        """Run agent loop with simulated thoughts and actions."""
        context = f"Question: {question}\\n\\n"
        trajectory = []
        
        for iteration in range(min(self.max_iterations, len(simulated_thoughts))):
            # Step 1: Thought
            thought = simulated_thoughts[iteration]
            context += f"Thought: {thought}\\n"
            
            # Step 2: Action
            action_str = simulated_actions[iteration]
            context += f"Action: {action_str}\\n"
            
            # Parse action
            action_type, param = self._parse_action(action_str)
            
            # Check for termination
            if action_type == "finish":
                context += f"Final Answer: {param}\\n"
                trajectory.append({
                    "iteration": iteration,
                    "thought": thought,
                    "action": action_str,
                    "observation": param,
                    "type": "finish"
                })
                break
            
            # Step 3: Observation
            observation = self._execute_action(action_type, param)
            context += f"Observation: {observation}\\n\\n"
            
            trajectory.append({
                "iteration": iteration,
                "thought": thought,
                "action": action_str,
                "observation": observation,
                "type": action_type
            })
        
        return {
            "question": question,
            "trajectory": trajectory,
            "num_iterations": len(trajectory),
            "context": context
        }
    
    def _parse_action(self, action_str: str) -> Tuple[str, str]:
        """Parse 'Search[query]' into ('search', 'query')."""
        match = re.match(r'(\w+)\\[(.+?)\\]', action_str.strip())
        if match:
            return match.group(1).lower(), match.group(2)
        return "unknown", action_str
    
    def _execute_action(self, action_type: str, param: str) -> str:
        """Execute action using tool."""
        if action_type in self.tools:
            return self.tools[action_type](param)
        return f"Error: Unknown action '{action_type}'"

# Define tools
def search_tool(query: str) -> str:
    """Simulate web search."""
    results = {
        "mechanical clock inventor": "John Harrison invented the marine chronometer in 1735",
        "john harrison birthplace": "John Harrison was born in Foulby, Yorkshire, England",
        "python syntax": "Python uses indentation for code blocks"
    }
    return results.get(query.lower(), f"No results found for '{query}'")

def lookup_tool(entity: str) -> str:
    """Look up entity."""
    lookups = {
        "john harrison": "English horologist (clockmaker), 1693-1776"
    }
    return lookups.get(entity.lower(), f"Unknown entity: {entity}")

tools = {"search": search_tool, "lookup": lookup_tool}

# Run example agent
agent = SimpleReActAgent(tools, max_iterations=10)

question = "What is the birthplace of the inventor of the mechanical clock?"
thoughts = [
    "I need to find who invented the mechanical clock first.",
    "Now I know it's John Harrison. Let me find his birthplace.",
    "Great! I have the answer."
]
actions = [
    "Search[mechanical clock inventor]",
    "Search[john harrison birthplace]",
    "Finish[Foulby, Yorkshire, England]"
]

result = agent.run(question, thoughts, actions)

print("ReAct Agent Trajectory:")
print("=" * 60)
for step in result["trajectory"]:
    print(f"Step {step['iteration'] + 1}:")
    print(f"  Thought: {step['thought']}")
    print(f"  Action: {step['action']}")
    print(f"  Observation: {step['observation']}")
    print()

print(f"Total iterations: {result['num_iterations']}")

## Level 2: Advanced ReAct with Error Handling

Robust agent that handles invalid actions, loops, and graceful degradation.

In [ ]:
class RobustReActAgent:
    """ReAct agent with error handling and loop detection."""
    
    def __init__(self, tools: Dict, max_iterations: int = 10):
        self.tools = tools
        self.max_iterations = max_iterations
        self.failed_actions = {}  # Track failed actions
    
    def run(self, question: str, scenarios: List[Dict]) -> Dict:
        """Run agent with multiple scenarios demonstrating error handling."""
        results = []
        
        for scenario in scenarios:
            context = f"Question: {scenario['question']}\\n\\n"
            trajectory = []
            failed_actions_local = set()
            
            for iteration in range(min(self.max_iterations, len(scenario['actions']))):
                thought = scenario['thoughts'][iteration]
                action_str = scenario['actions'][iteration]
                
                context += f"Thought: {thought}\\n"
                context += f"Action: {action_str}\\n"
                
                action_type, param = self._parse_action(action_str)
                
                # Termination
                if action_type == "finish":
                    context += f"Final Answer: {param}\\n"
                    trajectory.append({
                        "thought": thought,
                        "action": action_str,
                        "observation": f"Answer: {param}",
                        "status": "success"
                    })
                    break
                
                # Validation and execution
                if action_type == "unknown":
                    observation = f"Error: Invalid action format. Expected: ActionType[param]"
                    status = "error_format"
                elif action_type not in self.tools:
                    observation = f"Error: Unknown action '{action_type}'. Available: {', '.join(self.tools.keys())}"
                    status = "error_unknown"
                elif (action_type, param) in failed_actions_local:
                    observation = f"Note: Already tried {action_type}[{param}] and it failed. Try different approach."
                    status = "repeat_failure"
                else:
                    observation = self.tools[action_type](param)
                    if "error" in observation.lower():
                        failed_actions_local.add((action_type, param))
                        status = "action_error"
                    else:
                        status = "success"
                
                context += f"Observation: {observation}\\n\\n"
                trajectory.append({
                    "thought": thought,
                    "action": action_str,
                    "observation": observation,
                    "status": status
                })
            
            results.append({
                "scenario": scenario['name'],
                "question": scenario['question'],
                "trajectory": trajectory,
                "num_steps": len(trajectory),
                "success": trajectory[-1]['status'] == 'success' if trajectory else False
            })
        
        return results
    
    def _parse_action(self, action_str: str) -> Tuple[str, str]:
        match = re.match(r'(\w+)\\[(.+?)\\]', action_str.strip())
        if match:
            return match.group(1).lower(), match.group(2)
        return "unknown", action_str

# Test scenarios
agent = RobustReActAgent(tools)

scenarios = [
    {
        "name": "Happy Path",
        "question": "What is Python?",
        "thoughts": [
            "I should search for Python definition.",
            "Got the definition. I can answer now."
        ],
        "actions": [
            "Search[python syntax]",
            "Finish[Python is a programming language]"
        ]
    },
    {
        "name": "Error Recovery",
        "question": "Who is Alice?",
        "thoughts": [
            "Try invalid action first.",
            "Use valid action instead.",
            "Got answer."
        ],
        "actions": [
            "InvalidAction[Alice]",
            "Lookup[john harrison]",
            "Finish[John Harrison was a clockmaker]"
        ]
    },
    {
        "name": "Loop Detection",
        "question": "Find unknown info",
        "thoughts": [
            "Search for something.",
            "Try same search again (loop).",
            "Use different approach."
        ],
        "actions": [
            "Search[unknown entity]",
            "Search[unknown entity]",
            "Finish[Not found]"
        ]
    }
]

results = agent.run("demo", scenarios)

print("ReAct Error Handling Scenarios:")
print("=" * 60)
for result in results:
    print(f"\\nScenario: {result['scenario']}")
    print(f"Success: {result['success']}")
    print(f"Steps: {result['num_steps']}")
    for i, step in enumerate(result['trajectory'], 1):
        status_icon = "✓" if step['status'] == 'success' else "✗" if 'error' in step['status'] else "→"
        print(f"  [{status_icon}] {step['action'][:30]:<30} → {step['status']}")

## Real-World Example 1: Question Answering with Tools

Building a fact-checking agent that searches and verifies information.

In [ ]:
# Enhanced tool set for QA tasks
knowledge_base = {
    "capital of france": "Paris",
    "population of paris": "2.2 million",
    "river in paris": "Seine River",
    "eiffel tower location": "Paris, France",
    "eiffel tower year": "1889"
}

def qa_search(query: str) -> str:
    """Search knowledge base for query."""
    for key, value in knowledge_base.items():
        if all(word in key for word in query.lower().split()):
            return f"Found: {key} = {value}"
    return f"No results for: {query}"

def qa_verify(claim: str, fact: str) -> str:
    """Verify if claim matches fact."""
    if fact.lower() in claim.lower():
        return f"Verified: Claim matches fact"
    return f"Mismatch: Claim '{claim}' does not match fact '{fact}'"

qa_tools = {"search": qa_search, "verify": qa_verify}

class QAReActAgent:
    def __init__(self, tools, max_iterations=5):
        self.tools = tools
        self.max_iterations = max_iterations
    
    def run(self, question: str, steps: List[Dict]) -> Dict:
        trajectory = []
        for i, step in enumerate(steps):
            if i >= self.max_iterations:
                break
            
            action_type, param = self._parse_action(step['action'])
            if action_type == "finish":
                trajectory.append({
                    "step": i + 1,
                    "action": step['action'],
                    "result": param
                })
                break
            
            if action_type in self.tools:
                result = self.tools[action_type](param)
            else:
                result = "Error"
            
            trajectory.append({
                "step": i + 1,
                "action": step['action'],
                "result": result
            })
        
        return {"question": question, "trajectory": trajectory}
    
    def _parse_action(self, action_str: str) -> Tuple[str, str]:
        match = re.match(r'(\w+)\\[(.+?)\\]', action_str.strip())
        if match:
            return match.group(1).lower(), match.group(2)
        return "unknown", action_str

# Run QA agent
qa_agent = QAReActAgent(qa_tools)

qa_steps = [
    {"action": "Search[capital of france]"},
    {"action": "Search[river in paris]"},
    {"action": "Finish[Paris is the capital of France. The Seine River flows through Paris."}
]

qa_result = qa_agent.run("Tell me about Paris, France", qa_steps)

print("QA Agent Trajectory:")
print("=" * 60)
for step in qa_result['trajectory']:
    print(f"Step {step['step']}: {step['action']}")
    print(f"  Result: {step['result']}")
    print()

## Real-World Example 2: Complex Multi-Step Reasoning

Agent solving a problem requiring multiple steps and backtracking.

In [ ]:
# Math problem solving agent
def math_calculate(expr: str) -> str:
    try:
        result = eval(expr)
        return f"Result: {result}"
    except:
        return f"Error in expression: {expr}"

math_tools = {"calculate": math_calculate}

class MathReActAgent:
    def __init__(self, tools, max_iterations=10):
        self.tools = tools
        self.max_iterations = max_iterations
    
    def solve(self, problem: str, reasoning_chain: List[Dict]) -> Dict:
        """Solve math problem with step-by-step reasoning."""
        steps = []
        
        for i, item in enumerate(reasoning_chain):
            if i >= self.max_iterations:
                break
            
            step = {
                "step": i + 1,
                "thought": item['thought'],
                "action": item['action']
            }
            
            # Parse and execute action
            match = re.match(r'(\w+)\\[(.+?)\\]', item['action'].strip())
            if match:
                action_type = match.group(1).lower()
                param = match.group(2)
                
                if action_type == "finish":
                    step['result'] = param
                    step['type'] = 'finish'
                elif action_type in self.tools:
                    step['result'] = self.tools[action_type](param)
                    step['type'] = 'action'
                else:
                    step['result'] = f"Unknown action: {action_type}"
                    step['type'] = 'error'
            else:
                step['result'] = "Invalid action format"
                step['type'] = 'error'
            
            steps.append(step)
        
        return {
            "problem": problem,
            "steps": steps,
            "num_steps": len(steps)
        }

math_agent = MathReActAgent(math_tools)

# Complex problem: "If John has $50, buys 3 items at $12 each, how much money is left?"
problem_reasoning = [
    {"thought": "John starts with $50.", "action": "Calculate[50]"},
    {"thought": "He buys 3 items at $12 each. Total spent: 3 * 12.", "action": "Calculate[3 * 12]"},
    {"thought": "Money left: 50 - 36.", "action": "Calculate[50 - 36]"},
    {"thought": "I have the answer.", "action": "Finish[John has $14 left]"}
]

math_result = math_agent.solve(
    "If John has $50, buys 3 items at $12 each, how much money is left?",
    problem_reasoning
)

print("Math Problem Solving with ReAct:")
print("=" * 60)
print(f"Problem: {math_result['problem']}")
print()
for step in math_result['steps']:
    print(f"Step {step['step']}: {step['thought']}")
    print(f"  Action: {step['action']}")
    print(f"  Result: {step['result']}")
    print()

## Comparison and Analysis

Visualizing agent efficiency, trajectory lengths, and success rates.

In [ ]:
# Simulate agent performance on different problem types
data = {
    "Simple lookup": {"cot_steps": 1, "react_steps": 2, "cot_success": 0.4, "react_success": 0.95},
    "Multi-step arithmetic": {"cot_steps": 3, "react_steps": 4, "cot_success": 0.6, "react_success": 0.90},
    "Complex reasoning": {"cot_steps": 5, "react_steps": 6, "cot_success": 0.45, "react_success": 0.85},
    "Knowledge-intensive": {"cot_steps": 4, "react_steps": 5, "cot_success": 0.25, "react_success": 0.92},
    "Tool-dependent": {"cot_steps": 5, "react_steps": 5, "cot_success": 0.10, "react_success": 0.88}
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Steps required
problems = list(data.keys())
cot_steps = [data[p]["cot_steps"] for p in problems]
react_steps = [data[p]["react_steps"] for p in problems]
x = np.arange(len(problems))
width = 0.35

axes[0, 0].bar(x - width/2, cot_steps, width, label="CoT Only", color="steelblue", alpha=0.8)
axes[0, 0].bar(x + width/2, react_steps, width, label="ReAct", color="darkorange", alpha=0.8)
axes[0, 0].set_ylabel("Steps Needed", fontweight="bold")
axes[0, 0].set_title("Iteration Count by Problem Type", fontweight="bold")
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(problems, rotation=45, ha="right", fontsize=9)
axes[0, 0].legend()
axes[0, 0].grid(axis="y", alpha=0.3)

# Plot 2: Success rate
cot_success = [data[p]["cot_success"] * 100 for p in problems]
react_success = [data[p]["react_success"] * 100 for p in problems]

axes[0, 1].bar(x - width/2, cot_success, width, label="CoT Only", color="steelblue", alpha=0.8)
axes[0, 1].bar(x + width/2, react_success, width, label="ReAct", color="darkorange", alpha=0.8)
axes[0, 1].set_ylabel("Success Rate (%)", fontweight="bold")
axes[0, 1].set_title("Accuracy by Problem Type", fontweight="bold")
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(problems, rotation=45, ha="right", fontsize=9)
axes[0, 1].legend()
axes[0, 1].set_ylim([0, 100])
axes[0, 1].grid(axis="y", alpha=0.3)

# Plot 3: Steps vs Success (scatter)
all_cot_steps = [s for s in cot_steps for _ in range(1)]
all_cot_success = [s for s in cot_success for _ in range(1)]
all_react_steps = [s for s in react_steps for _ in range(1)]
all_react_success = [s for s in react_success for _ in range(1)]

axes[1, 0].scatter(all_cot_steps, all_cot_success, s=200, alpha=0.6, color="steelblue", label="CoT", edgecolors="black")
axes[1, 0].scatter(all_react_steps, all_react_success, s=200, alpha=0.6, color="darkorange", label="ReAct", edgecolors="black")
axes[1, 0].set_xlabel("Iteration Count", fontweight="bold")
axes[1, 0].set_ylabel("Success Rate (%)", fontweight="bold")
axes[1, 0].set_title("Efficiency: Success vs Iterations", fontweight="bold")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_ylim([0, 105])

# Plot 4: Summary metrics
metrics = ["Avg Steps", "Avg Success", "Max Steps", "Min Success"]
cot_metrics = [
    np.mean(cot_steps),
    np.mean(cot_success),
    max(cot_steps),
    min(cot_success)
]
react_metrics = [
    np.mean(react_steps),
    np.mean(react_success),
    max(react_steps),
    min(react_success)
]

x_metrics = np.arange(len(metrics))
axes[1, 1].bar(x_metrics - width/2, cot_metrics, width, label="CoT", color="steelblue", alpha=0.8)
axes[1, 1].bar(x_metrics + width/2, react_metrics, width, label="ReAct", color="darkorange", alpha=0.8)
axes[1, 1].set_ylabel("Value", fontweight="bold")
axes[1, 1].set_title("Aggregate Metrics", fontweight="bold")
axes[1, 1].set_xticks(x_metrics)
axes[1, 1].set_xticklabels(metrics, fontsize=9)
axes[1, 1].legend()
axes[1, 1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

print("\\nAgent Comparison Summary:")
print("=" * 60)
print(f"CoT: Avg {np.mean(cot_steps):.1f} steps, {np.mean(cot_success):.0f}% success")
print(f"ReAct: Avg {np.mean(react_steps):.1f} steps, {np.mean(react_success):.0f}% success")
print(f"\\nReAct improvement: +{np.mean(react_success) - np.mean(cot_success):.0f}% accuracy")
print(f"Cost: +{np.mean(react_steps) - np.mean(cot_steps):.1f} steps on average")

## Key Takeaways

**ReAct Loop:** Thought → Action → Observation → Repeat

### When to Use ReAct
- Knowledge-intensive tasks (need retrieval)
- Tool-dependent problems (need to use calculators, APIs)
- Multi-step problems (need to break down and verify)
- When reasoning alone fails (CoT accuracy < 60%)

### When to Use CoT Only
- Pure reasoning tasks (math, logic)
- Fast inference needed (skip tool calls)
- No external tools available
- Simple lookup questions

### Production Lessons
- **Iterations matter:** ReAct adds 1-4 steps per query
- **Tool quality matters:** Bad tools → bad agent
- **Loop detection:** Monitor for infinite loops (max 10-15 iterations)
- **Error handling:** Gracefully recover from invalid actions
- **Observation clarity:** Clear observations → better reasoning

### Key Insight
Combining reasoning with actions unlocks capabilities neither has alone. CoT can't verify facts; pure tools can't plan. ReAct does both.

### Next Steps
- [Chain-of-Thought](./01-chain-of-thought.md) — Understanding reasoning in agents
- [Tree of Thoughts](./03-tree-of-thoughts.md) — Exploring multiple action sequences